# Orders Pipeline Report

Automated data quality check and revenue summary report.

This notebook is executed by the `notebook_executor` Airflow DAG via Colab Enterprise.
It queries the monitoring lab pipeline tables and produces a summary of:
- Data freshness and row counts
- Null/invalid data checks
- Revenue summary by product and region
- Order status distribution

**Parameters** (injected by Airflow):
- `project_id`: GCP project ID
- `date`: Execution date (YYYY-MM-DD)

In [ ]:
# ---------------------------------------------------------------------------
# Parameters — injected by Airflow at execution time
# These defaults allow the notebook to run interactively for testing.
# ---------------------------------------------------------------------------
project_id = "da-monitoring-lab-07230757"  # overridden by Airflow
date = "2026-07-27"  # overridden by Airflow

# Derived constants
SILVER_DATASET = f"{project_id}.monitoring_lab_silver"
DATAMART_DATASET = f"{project_id}.monitoring_lab_datamart"

In [ ]:
from google.cloud import bigquery
from datetime import datetime, timezone

client = bigquery.Client(project=project_id)
print(f"Report for project: {project_id}")
print(f"Report date: {date}")
print(f"Generated at: {datetime.now(timezone.utc).isoformat()}")
print(f"Silver dataset: {SILVER_DATASET}")
print(f"Datamart dataset: {DATAMART_DATASET}")

## 1. Data Quality — Silver Layer

Check row counts, data freshness, and null values in the silver orders table.

In [ ]:
# ---------------------------------------------------------------------------
# Silver layer: row count, freshness, and null checks
# ---------------------------------------------------------------------------
dq_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(order_id IS NULL) AS null_order_ids,
    COUNTIF(customer_id IS NULL) AS null_customer_ids,
    COUNTIF(product IS NULL OR product = '') AS empty_products,
    COUNTIF(total_price IS NULL OR total_price <= 0) AS invalid_prices,
    COUNTIF(status IS NULL OR status = '') AS empty_statuses,
    MIN(created_at) AS earliest_order,
    MAX(created_at) AS latest_order,
    MAX(_ingested_at) AS last_ingestion,
    TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), MAX(_ingested_at), MINUTE) AS minutes_since_last_ingest
FROM `{SILVER_DATASET}.orders`
"""

dq_result = client.query(dq_query).to_dataframe()
print("=== Silver Layer Data Quality ===")
print(dq_result.to_string(index=False))

# Flag freshness issues
minutes_stale = dq_result['minutes_since_last_ingest'].iloc[0]
if minutes_stale and minutes_stale > 60:
    print(f"\n⚠️  WARNING: Data is {minutes_stale:.0f} minutes stale (threshold: 60 min)")
else:
    print(f"\n✅ Data freshness OK ({minutes_stale:.0f} min since last ingest)")

## 2. Revenue Summary — Datamart Layer

Top products by revenue from the datamart aggregation.

In [ ]:
# ---------------------------------------------------------------------------
# Datamart: revenue by product (top 10)
# ---------------------------------------------------------------------------
revenue_query = f"""
SELECT
    product,
    region,
    SUM(order_count) AS total_orders,
    SUM(total_quantity) AS total_quantity,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value
FROM `{DATAMART_DATASET}.revenue_by_product`
GROUP BY product, region
ORDER BY total_revenue DESC
LIMIT 10
"""

revenue_df = client.query(revenue_query).to_dataframe()
print("=== Top 10 Product-Region Combinations by Revenue ===")
print(revenue_df.to_string(index=False))
print(f"\nTotal revenue across all products: ${revenue_df['total_revenue'].sum():,.2f}")

In [ ]:
# ---------------------------------------------------------------------------
# Datamart: order status distribution
# ---------------------------------------------------------------------------
status_query = f"""
SELECT
    status,
    SUM(order_count) AS total_orders,
    ROUND(SUM(total_value), 2) AS total_value,
    ROUND(AVG(avg_processing_minutes), 1) AS avg_processing_min
FROM `{DATAMART_DATASET}.order_status_summary`
GROUP BY status
ORDER BY total_orders DESC
"""

status_df = client.query(status_query).to_dataframe()
print("=== Order Status Distribution ===")
print(status_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Summary report — structured output for monitoring
# ---------------------------------------------------------------------------
import json

summary = {
    "report_date": date,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "project_id": project_id,
    "data_quality": {
        "total_rows": int(dq_result['total_rows'].iloc[0]),
        "null_order_ids": int(dq_result['null_order_ids'].iloc[0]),
        "null_customer_ids": int(dq_result['null_customer_ids'].iloc[0]),
        "empty_products": int(dq_result['empty_products'].iloc[0]),
        "invalid_prices": int(dq_result['invalid_prices'].iloc[0]),
        "minutes_since_last_ingest": float(dq_result['minutes_since_last_ingest'].iloc[0] or 0),
        "freshness_ok": bool(minutes_stale is not None and minutes_stale <= 60),
    },
    "revenue": {
        "total_revenue": float(revenue_df['total_revenue'].sum()) if not revenue_df.empty else 0,
        "top_product": revenue_df['product'].iloc[0] if not revenue_df.empty else "N/A",
        "product_count": int(revenue_df['product'].nunique()) if not revenue_df.empty else 0,
    },
    "status_distribution": status_df.set_index('status')['total_orders'].to_dict() if not status_df.empty else {},
}

print("=== Report Summary (JSON) ===")
print(json.dumps(summary, indent=2, default=str))
print("\n✅ Orders pipeline report complete.")

---

## Report Metadata

- **Executed by**: `notebook_executor` Airflow DAG
- **Runtime**: Colab Enterprise (Vertex AI)
- **Source tables**: `monitoring_lab_silver.orders`, `monitoring_lab_datamart.revenue_by_product`, `monitoring_lab_datamart.order_status_summary`
- **Failure alerting**: Global failure listener → Cloud Logging → Cloud Monitoring alerts